## Emergency Medicine Triage RAG System (retrieve specific guideline)

This notebook implements a complete Retrieval-Augmented Generation (RAG) system for emergency medicine triage decision support. The system processes new patient cases by:
1. Analyzing the case description
2. Retrieving relevant triage guidelines 
3. Generating a comprehensive triage assessment

****RAG implementation with Specific Guideline sections only**

In [ ]:
import json
import re
from typing import List, Dict, Any, Optional, Tuple, TypedDict, Literal
import os
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm
from tqdm.notebook import tqdm
import traceback

# Pydantic imports for structured output
from pydantic import BaseModel, Field, ValidationError

# LangChain imports
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# LangGraph imports
from langgraph.graph import StateGraph, END

## System Architecture Overview

This notebook implements a multi-stage RAG pipeline using a graph-based architecture. The key components are:

1. **Data Sources**:
   - Triage guidelines structured in sections

2. **LLM setup**:
   - **LLM Initialization** (choose one): Used to generate clinical summaries and reasoning tasks
        1. **DeepSeek-v3** (updated to deepseek-v4-pro)
        2. **gpt-4o**
        3. **claude-3-7-sonnet**

3. **Pipeline Flow**:
   - Case preprocessing and summarization
   - Guideline section retrieval and selection
   - Guideline-based final assessment
   
Each component is implemented as a node in a LangGraph that manages state transitions through the pipeline.

## Step 1: Initial Setup and Configuration

This section initializes all required components:

1. **Environment Configuration**:
   - Loading API keys and endpoints from environment variables
   
2. **LLM setup**:
   - **LLM Initialization** (choose one): Used to generate clinical summaries and reasoning tasks
        1. **DeepSeek-v3** (updated to deepseek-v4-pro)
        2. **gpt-4o**
        3. **claude-3-7-sonnet**

   > **Note:** Uncomment exactly one of the three LLM initialization cells below. The chosen LLM also sets the `model` variable, which is used in output file and directory naming.

3. **Data Loading**:
   - Triage guidelines with section summaries

4. **State Definition**:
   - Defining the `GraphState` class to track data through the pipeline
   - Managing transitions between processing stages

In [ ]:
# Load environment variables
load_dotenv()

# Verify environment variables
required_env_vars = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "DEEPSEEK_API_KEY",
    "AZURE_OPENAI_LLM_DEPLOYMENT_NAME",
    "AZURE_OPENAI_API_VERSION",
    "ANTHROPIC_API_KEY",
    "ANTHROPIC_MODEL_NAME",
]

missing_vars = [var for var in required_env_vars if not os.getenv(var)]
if missing_vars:
    raise ValueError(f"Missing environment variables: {', '.join(missing_vars)}")

print("Environment variables loaded successfully.")

In [ ]:
from langchain_deepseek import ChatDeepSeek

# Initialize LLM
llm = ChatDeepSeek(
    model="deepseek-v4-pro",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    temperature=0,
    # Move extra_body here directly:
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)
print("LLM initialized successfully.")

model = "deepseek-v4-pro"

In [ ]:
# from langchain_openai import AzureChatOpenAI


# def initialize_azure_openai(
#     azure_deployment_name: str = os.getenv("AZURE_OPENAI_LLM_DEPLOYMENT_NAME"),
#     azure_endpoint: str = os.getenv("AZURE_OPENAI_ENDPOINT"),
#     azure_api_key: str = os.getenv("AZURE_OPENAI_API_KEY"),
#     azure_api_version: str = os.getenv("AZURE_OPENAI_API_VERSION"),
#     temperature: float = 0
# ):
#     """
#     Initialize Azure OpenAI client
    
#     Args:
#         azure_deployment_name (str): Azure deployment name
#         azure_endpoint (str): Azure endpoint URL
#         azure_api_key (str): Azure API key
#         azure_api_version (str): Azure API version
#         temperature (float): Temperature for generation
        
#     Returns:
#         AzureChatOpenAI: Initialized Azure OpenAI client
#     """
#     return AzureChatOpenAI(
#         azure_deployment=azure_deployment_name,
#         azure_endpoint=azure_endpoint,
#         api_key=azure_api_key,
#         api_version=azure_api_version,
#         temperature=temperature
#     )


# llm = initialize_azure_openai()

# print("LLM initialized successfully.")

# model = "gpt-4o"

In [ ]:
# from langchain_anthropic import ChatAnthropic

# def initialize_claude(
#     api_key: str = os.getenv("ANTHROPIC_API_KEY"),
#     model_name: str = os.getenv("ANTHROPIC_MODEL_NAME"),
#     temperature: float = 0,
#     max_tokens: int = 8192,  
#     streaming: bool = True
# ):
#     """
#     Initialize Claude client using LangChain
    
#     Args:
#         api_key (str): Anthropic API key
#         model_name (str): Model name to use (claude-3-7-sonnet-20250219 for latest Claude 3.7 Sonnet)
#         temperature (float): Temperature for generation
        
#     Returns:
#         ChatAnthropic: Initialized Claude client
#     """
#     return ChatAnthropic(
#         model=model_name,
#         anthropic_api_key=api_key,
#         temperature=temperature,
#         max_tokens=max_tokens,
#         streaming=streaming
#     )

# llm = initialize_claude()

# print("Claude LLM initialized successfully.")

# model = "claude-3-7"

In [ ]:
# Load the triage guidelines from the JSON file
def load_triage_guidelines(file_path) -> List[Dict[str, str]]:
    """
    Load the triage guidelines from a JSON file.
    
    Args:
        file_path (str): Path to the JSON file
        
    Returns:
        List[Dict[str, str]]: List of guideline sections with title, content, and summary
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        guidelines = json.load(f)
    
    print(f"Loaded {len(guidelines)} sections from {file_path}")
    return guidelines

# Load the guidelines
triage_guidelines = load_triage_guidelines(file_path=f"../../db/triage_sections_with_summaries_{model}.json")

# Extract just the metadata (title and summary) for retrieval decisions
guideline_section_metadata = [
    {"section_title": section["section_title"], "summary": section["summary"]}
    for section in triage_guidelines
]

In [ ]:
# Define state structure
class GraphState(dict):
    """State for the retrieval graph."""
    summary_text: Optional[str] = None              # case summary for guideline retrieval
    case_json_full: Dict[str, Any]                   # full case json
    guideline_section_metadata: List[Dict[str, str]] # guideline section metadata: title, summary
    selected_guideline_sections: Optional[List[str]] = None          # selected sections
    retrieved_guideline_content: Optional[List[Dict[str, str]]] = None  # selected sections content
    guideline_reasoning: Optional[str] = None        # reasoning for guideline selection
    final_assessment: Optional[str] = None           # final assessment output
    final_category: Optional[str] = None             # final triage category
    final_confidence: Optional[str] = None           # final confidence level
    processing_failed: Optional[bool] = False        # whether processing failed
    failure_reason: Optional[str] = None             # reason for failure
    failed_node: Optional[str] = None                # node that failed

In [ ]:
# --- Pydantic schemas for structured output ---

class GuidelineRetrievalOutput(BaseModel):
    """Structured output for guideline retrieval decision"""
    selected_sections: List[str] = Field(
        description="List of section_title(s) exactly as they appear in the guidelines, maximum 2",
        max_items=2
    )
    reasoning: str = Field(description="Brief explanation for section selection")

class TriageStep(BaseModel):
    """Individual step in triage prediction process"""
    category: Literal["1", "2", "3", "4", "5", "Not specified"] = Field(description="Triage category")
    confidence: Literal["High", "Medium", "Low"] = Field(description="Confidence level")
    reason: str = Field(description="Reasoning for this step")

class TriagePredictionOutput(BaseModel):
    """Structured output for guideline-based triage prediction (2-step)"""
    step1_clinical_risk: TriageStep = Field(description="Clinical risk assessment")
    step2_guidelines: TriageStep = Field(description="Guidelines-based assessment")
    final_decision: TriageStep = Field(description="Final triage decision")

# --- Token usage tracking ---

class TokenUsageMetrics(BaseModel):
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0

def get_token_usage(response) -> TokenUsageMetrics:
    try:
        if hasattr(response, 'response_metadata') and 'token_usage' in response.response_metadata:
            usage = response.response_metadata['token_usage']
            return TokenUsageMetrics(
                prompt_tokens=usage.get('prompt_tokens', 0),
                completion_tokens=usage.get('completion_tokens', 0),
                total_tokens=usage.get('total_tokens', 0)
            )
    except Exception as e:
        print(f"Could not extract token usage: {e}")
    return TokenUsageMetrics()

def execute_structured_node_with_retry(node_func, state, node_name, max_retries=3):
    """
    Execute a node with retry logic for malformed structured responses.
    
    Args:
        node_func: The node function to execute
        state: Current graph state
        node_name: Name of the node (for logging)
        max_retries: Maximum number of retry attempts
        
    Returns:
        Updated state or state with failure flag
    """
    for attempt in range(max_retries):
        try:
            print(f"\n=== {node_name}: Attempt {attempt + 1}/{max_retries} ===")
            result = node_func(state)
            print(f"{node_name} succeeded on attempt {attempt + 1}")
            return result
        except (ValidationError, ValueError, KeyError, json.JSONDecodeError) as e:
            print(f"{node_name} attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                print(f"{node_name} FAILED after {max_retries} retries")
                failed_state = state.copy()
                failed_state["processing_failed"] = True
                failed_state["failure_reason"] = f"{node_name} failed after {max_retries} retries: {str(e)}"
                failed_state["failed_node"] = node_name
                return failed_state
        except Exception as e:
            print(f"{node_name} attempt {attempt + 1} failed with unexpected error: {e}")
            if attempt == max_retries - 1:
                failed_state = state.copy()
                failed_state["processing_failed"] = True
                failed_state["failure_reason"] = f"{node_name} failed with unexpected error: {str(e)}"
                failed_state["failed_node"] = node_name
                return failed_state
    return state

## Step 2: Utility Functions

These utility functions handle data processing tasks throughout the pipeline:

1. **Case Loading**:
   - Functions to load and preprocess case JSON files
   
2. **Metadata Filtering**:
   - Functions to filter vector database results by metadata
   - Enables more targeted retrieval of similar past cases

In [ ]:
# Initial loading of JSON file (new case)
def load_case_from_json(file_path: str) -> Dict[str, Any]:
    """
    Load a FULL case from a JSON file as NEW case. 

    Args:
        file_path (str): Path to the JSON file

    Returns:
        Dict[str, Any]: FULL Case data, with "Case Disposition" removed(required for test/val set).
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        case_data = json.load(f)

    # Remove sections if it exists, required for ground_truth set 
    if "Case Disposition" in case_data:
        case_data.pop("Case Disposition")
    if "Ground Truth" in case_data:
        case_data.pop("Ground Truth")

    return case_data

## RAG Pipeline Implementation

The following sections implement the core RAG pipeline using a modular node-based approach. Each node performs a specific function and passes its results to subsequent nodes through the shared state object.

## Step 3: Node Definitions

Each node in the LangGraph represents a discrete processing step in the RAG pipeline. These nodes transform the input state and pass the results to downstream nodes.

### Core Pipeline Components

The pipeline consists of three main stages:
1. **Preprocessing**: Analyzing and summarizing the input case
2. **Retrieval**: Finding relevant guidelines
3. **Generation**: Producing the final triage assessment

### 3.1: Preprocessing Node
The preprocessing node converts the raw case data into a concise clinical summary that captures the most relevant medical information. This summary serves as the query for both guideline retrieval and case similarity search, ensuring that retrieval focuses on clinically significant elements.

In [ ]:
# Define node to preprocess case data
def process_case_summary(state: GraphState) -> GraphState:
    
    ## Create Summary for VectorDB retrieval 
    
    sections_to_expand = {}
    
    sections_for_summary = {}
    if 'Demographics' in state["case_json_full"]:
        if 'Age' in state["case_json_full"]['Demographics']:
            sections_for_summary['Age'] = state["case_json_full"].get("Demographics", {}).get("Age", "")
        if 'Sex' in state["case_json_full"]['Demographics']:
            sections_for_summary['Sex'] = state["case_json_full"].get("Demographics", {}).get("Sex", "")
    if 'Vitals and Observations' in state["case_json_full"]:
        sections_for_summary['Vitals and Observations'] = state["case_json_full"].get("Vitals and Observations", {})
    if "Clinical Presentation" in state["case_json_full"]:
        sections_for_summary['Clinical Presentation'] = state["case_json_full"].get("Clinical Presentation", {})
        
    summary_prompt = ChatPromptTemplate.from_messages([
        ("system", """
        You are a specialized medical summarization assistant for emergency triage notes. Your task is to analyze structured JSON data from triage notes and create a concise, retrievable summary highlighting only the most important clinical information.
        ## INPUT STRUCTURE
        You will receive structured JSON data containing:
        - Demographics (sex, age)
        - Clinical Presentation (chief complaint, condition on arrival)
        - Vitals and Observations

        ## OUTPUT REQUIREMENTS
        1. Generate a concise (2-3 sentence) summary focusing on:
           - Patient demographics (sex, age)
           - Key presenting complaints
           - Abnormal physical findings
           - ONLY abnormal vital signs according to strict criteria

        2. Write in a factual, medical style optimized for vector embedding retrieval.

        ## MEDICAL TERMINOLOGY GUIDELINES
        - Interpret medical abbreviations contextually 
        - Fix obvious typos/misspellings based on medical context
        - Maintain medical precision while expanding clarity
        - For Hong Kong hospital/facility abbreviations:
          - "PMH" → "Princess Margaret Hospital"
          - "KCH" → "Kwai Chung Hospital"
          - "OAH" → "Old Age Home"
          - If uncertain about any abbreviation, do NOT expand it

        ## VITAL SIGNS ABNORMALITY CRITERIA
        ONLY mention vital signs that meet these specific abnormality thresholds:

        - Blood Pressure (Adult):
          - Hypertension: SBP > 180 mmHg or DBP > 100 mmHg
          - Hypotension: SBP < 90 mmHg

        - Heart Rate (Adults):
          - Tachycardia: HR > 120/min
          - Bradycardia: HR < 50/min

        - Temperature (Adult):
          - Fever: Temp ≥ 38°C
          - Hypothermia: Temp < 35°C

        - Blood Sugar:
          - Hyperglycemia: H'stix > 16 mmol/l
          - Hypoglycemia: H'stix < 3.9 mmol/l

        - Respiratory Status (always consider):
          - Oxygen supplementation (if present)
          - SpO2 level if abnormal
          - Abnormal respiratory rate

        - Conscious Level:
          - GCS score if not 15/15
          - Pupil abnormalities (if provided)

        - Pregnancy/Menstruation: Note if pregnant or abnormal menstrual patterns

        ## EXAMPLES OF GOOD SUMMARIES:

        ### Example 1:
        Input: [
            {{
                "Demographics": {{
                    "Sex": "F",
                    "Age": "56 years"
                }},
                "Clinical Presentation": {{
                    "Chief complaint": "R/F ? IHD, tachycardia oedema\\nSOB , LL edema and neck swelling x 10/7 \\nchest discomfort - , palpitiation",
                    "Condition on Arrival": "speak in sentence"
                }},
                "Vitals and Observations": {{
                    "Vital Signs": {{
                        "GCS": "E: 4, V: 5, M: 6, Score: 15/15",
                        "BP": "1st: 144/89 mmHg",
                        "PR/AR": "99",
                        "Temp": "36.2",
                        "SpO2": "100% on room air, --",
                        "RR": "--",
                        "PFR": "--",
                        "Limbs": "--",
                        "Pupil": "--"
                    }},
                    "Triage Intervention": {{
                        "H'stix": "",
                        "H'cue": ""
                    }},
                    "Menstruation / Pregnancy Status": "unspecified"
                }}
            }}
        ]
        
        Output: "56-year-old female presenting with shortness of breath, lower limb edema, and neck swelling for 10 days, accompanied by chest discomfort and palpitations. Patient can speak in full sentences and has been referred for evaluation of possible ischemic heart disease with tachycardia and edema. All vital signs are within normal ranges."

        ### Example 2:
        Input: [
            {{
                "Demographics": {{
                    "Sex": "M",
                    "Age": "64 years"
                }},
                "Clinical Presentation": {{
                    "Chief complaint": "SOB this morning 0730\\nroom air 70%",
                    "Condition on Arrival": "no lower limb oedema\\nsputum sound+/-"
                }},    
                "Vitals and Observations": {{
                    "Vital Signs": {{
                        "GCS": "",
                        "BP": "1st: 165/97 mmHg",
                        "PR/AR": "130",
                        "Temp": "38.1",
                        "SpO2": "96% on 6 L/min via Nasal Cannula",
                        "RR": "20",
                        "PFR": "--",
                        "Limbs": "--",
                        "Pupil": "--"
                    }},
                    "Triage Intervention": {{
                        "H'stix": "",
                        "H'cue": ""
                    }}
                }}
            }}
        ]
        Output: "64-year-old male presenting with sudden onset shortness of breath at 7:30 AM with significant hypoxia (70% on room air). Patient has tachycardia (HR 130), fever (38.1°C), and requires oxygen supplementation (6 L/min via Nasal Cannula). Clinical examination reveals sputum sounds without lower limb edema."

        ## IMPORTANT REMINDERS:
        - Focus on clinical relevance and retrieval effectiveness
        - Maintain medical accuracy while being concise
        - Only highlight abnormal vitals based on specific thresholds
        - Fix obvious errors but maintain meaning integrity
        - Adjust abnormality thresholds for pediatric cases
        """),
        ("user", """
        Case Data (JSON):
        {case_data}

        OUTPUT (Summary):
        """)
    ])
        
    # Prepare the input for the prompt
    case_data_str = json.dumps(sections_for_summary, indent=2)
    input_values = {
        "case_data": case_data_str
    }
    # Get the LLM response
    chain = summary_prompt | llm | StrOutputParser()
    response = chain.invoke(input_values)
    
    updated_state = state.copy()
    updated_state["summary_text"] = response
    return updated_state

### 3.2 Retrieval Nodes

The retrieval nodes perform three key functions:
1. **Guideline selection**: Identifying the most relevant guideline sections for the current case

In [ ]:
def guideline_retrieval_decision(state: GraphState) -> GraphState:
    """
    Optimized guideline retrieval with structured output focusing on chief complaint and abnormal vitals.
    Limits to 1-2 most relevant sections.
    """
    
    summary_text = state["summary_text"]
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an emergency triage assistant. Your task is to select ONLY the 1-2 most critical guideline sections based on the chief complaint and abnormal vital signs.

        CRITICAL SELECTION RULES:
        1. Focus ONLY on the PRIMARY presenting complaint and documented ABNORMAL vital signs
        2. Select MAXIMUM 2 sections - prioritize life-threatening conditions first
        3. Do NOT select sections for normal findings or assumptions
        4. Consider abnormal vitals ONLY if they meet these thresholds:
           - BP: SBP > 180 or < 90 mmHg, DBP > 100 mmHg
           - HR: > 120 or < 50 (adults)
           - Temp: ≥ 38°C or < 35°C
           - Blood sugar: > 16 or < 3.9 mmol/l
           - SpO2: < 95% or requiring oxygen support
           - GCS: < 15

        SELECTION PRIORITY:
        1. Life-threatening symptoms (chest pain, SOB, altered consciousness)
        2. Abnormal vital signs meeting thresholds above
        3. Specific documented complaints
        4. If no specific sections apply, return an empty list.
        """),
        ("user", """
        Clinical Summary: {summary_text}
        
        Available Guideline Sections:
        {guideline_section_metadata}
        """)
    ])
    
    input_values = {
        "summary_text": summary_text,
        "guideline_section_metadata": "\n".join([
            f"Section: {meta['section_title']}\nSummary: {meta['summary']}\n"
            for meta in state["guideline_section_metadata"]
        ])
    }
    
    structured_llm = llm.with_structured_output(GuidelineRetrievalOutput)
    chain = prompt | structured_llm

    MAX_RETRIES = 3
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = chain.invoke(input_values)
            break
        except Exception as e:
            last_error = e
            print(f"[guideline_retrieval_decision retry {attempt}/{MAX_RETRIES}] {type(e).__name__}: {e}")
            if attempt == MAX_RETRIES:
                raise last_error

    print(f"Selected sections: {response.selected_sections}")
    print(f"Reasoning: {response.reasoning}")
    
    updated_state = state.copy()
    updated_state["selected_guideline_sections"] = response.selected_sections
    updated_state["guideline_reasoning"] = response.reasoning
    return updated_state


def retrieved_guideline_section_content(state: GraphState) -> GraphState:
    """
    Retrieve the full content of the selected sections.
    
    Args:
        state: Current state with selected section titles
        
    Returns:
        Updated state with retrieved section content
    """
    selected_guideline_sections = state["selected_guideline_sections"]
    full_guidelines = triage_guidelines
    
    retrieved_guideline_content = []
    for section in full_guidelines:
        if section["section_title"] in selected_guideline_sections:
            retrieved_guideline_content.append(section)
    
    print(f"Retrieved {len(retrieved_guideline_content)} guideline section(s)")
    updated_state = state.copy()
    updated_state["retrieved_guideline_content"] = retrieved_guideline_content
    return updated_state

### 3.3 Prediction node

The prediction node integrates all retrieved information to generate a structured triage assessment. It follows a stepwise approach that:
1. CLINICAL RISK ASSESSMENT (Based on Triage Definitions)
2. GUIDELINE-BASED ASSESSMENT (Medical Condition Specific)
3. FINAL DECISION

In [ ]:
def guideline_based_triage_prediction(state: GraphState) -> GraphState:
    """
    Guideline-based triage prediction using only triage definitions and retrieved guidelines.
    """
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an emergency triage specialist. Use this 2-step approach to determine the appropriate triage category based on triage definitions and specific medical guidelines only.

        TRIAGE CATEGORY DEFINITIONS:
        | Triage Category | Patient Conditions | Actions of Staff | Target Response Time (Time from registration to medical consultation) |
        |---------------|------------------|----------------|---------------------------------------------------------------------|
        | 1 Critical | • Suffer from a life-threatening condition(s) caused by a major event • With unstable vital signs requiring immediate resuscitation | • Direct patient to resuscitation room • Attend patient immediately by a team comprising medical and nursing staff | • Immediate • 100% of cases within the target response time |
        | 2 Emergency | • Suffer from a potentially life-threatening condition • Borderline vital signs but with potential risk of rapid deterioration • Require emergency treatment and immediate continuous close monitoring | • Direct patient to resuscitation room / treatment cubicle • Offer medical attention and immediate continuous close monitoring within 15 mins. | • < 15 mins • 95% of cases within the target response time |
        | 3 Urgent | • Suffer from a major condition with potential risk of deterioration • Stable vital signs | • Direct patient to cubicle | • < 30 mins • 90% of cases within the target response time |
        | 4 Semi-urgent | • Suffer from acute but stable condition(s) • Stable vital signs • Can afford to wait some time without serious complications | • Direct patient to cubicle / walk-in clinic | • < 120 mins |
        | 5 Non-urgent | • Suffer from minor and stable condition(s) (including acute and non-acute conditions) • Can afford to wait without deterioration • Stable vital signs | • Direct patient to walk-in clinic | Remarks: • Conditions can be treated in primary health care facilities • Should be based on clinical judgement only. Economic, social factors and availability of facilities should not be taken into consideration. |

        USE THIS EXACT 2-STEP FORMAT:

        ### STEP 1: CLINICAL RISK ASSESSMENT (Based on Triage Definitions)
        **Focus:** Determine overall CLINICAL RISK and URGENCY based on triage category definitions
        **Patient Profile:** [Age, sex, key symptoms, abnormal vitals only]
        **Risk of Deterioration:** [Immediate/High/Moderate/Low - based on clinical instability]
        **Urgency Level:** [Life-threatening/Potentially life-threatening/Major/Acute/Minor]
        **Initial Triage Category (Clinical Risk):** [1/2/3/4/5]

        ### STEP 2: GUIDELINE-BASED ASSESSMENT (Medical Condition Specific)
        **Focus:** Apply specific medical guidelines for documented conditions
        **Most Relevant Guideline:** [Title of guideline, or "None specific"]
        **Specific Medical Condition:** [Condition addressed by guideline]
        **Guideline Criteria:** [Specific criteria from guideline]
        **Criteria Met:** [Yes/No with evidence]
        **Guideline-Recommended Category:** [1/2/3/4/5, or "Not specified"]

        ### FINAL DECISION
        **Step 1 Category (Clinical Risk):** [1/2/3/4/5]
        **Step 2 Category (Guidelines):** [1/2/3/4/5 or "Not specified"]
        **FINAL TRIAGE CATEGORY:** [1/2/3/4/5]
        **CONFIDENCE:** [High/Medium/Low]
        **KEY RATIONALE:** [Explain which step(s) drove the final decision]
        
        CRITICAL INSTRUCTIONS:
        - Step 1: Focus purely on clinical urgency and risk of deterioration per triage definitions
        - Step 2: Apply medical guidelines for specific documented conditions only
        """),
        ("user", """
        Current Case:
        {case_summary}
        
        Relevant Guidelines:
        {guideline_content}
        
        Please provide your 2-step triage assessment:
        """)
    ])
    
    case_summary = f"""
    Clinical Summary: {state['summary_text']}
    
    Demographics: {state['case_json_full'].get('Demographics', {})}
    """
    
    guideline_content = ""
    if state["retrieved_guideline_content"]:
        for section in state["retrieved_guideline_content"]:
            guideline_content += f"GUIDELINE: {section['section_title']}\n{section['content']}\n\n"
    else:
        guideline_content = "No specific guidelines retrieved."
    
    input_values = {
        "case_summary": case_summary,
        "guideline_content": guideline_content
    }

    structured_llm = llm.with_structured_output(TriagePredictionOutput)
    chain = prompt | structured_llm
    response = chain.invoke(input_values)
    
    print(f"Step 1 (Clinical Risk): {response.step1_clinical_risk.category} - {response.step1_clinical_risk.confidence}")
    print(f"Step 2 (Guidelines): {response.step2_guidelines.category} - {response.step2_guidelines.confidence}")
    print(f"FINAL DECISION: {response.final_decision.category} - {response.final_decision.confidence}")
    
    updated_state = state.copy()
    updated_state["final_assessment"] = response.model_dump()
    updated_state["final_category"] = response.final_decision.category
    updated_state["final_confidence"] = response.final_decision.confidence
    
    return updated_state

### Pipeline Assembly

The following functions assemble the individual nodes into a complete processing pipeline using LangGraph. The graph structure defines:
1. The execution order of processing steps
2. Data flow between components
3. State management throughout the pipeline

In [ ]:
def build_guideline_only_triage_graph():
    """
    Build a guideline-only triage assessment graph (past cases removed).
    
    Returns:
        The constructed graph
    """
    graph = StateGraph(GraphState)
    
    guideline_prediction_with_retry = lambda state: execute_structured_node_with_retry(
        guideline_based_triage_prediction, state, "guideline_based_triage_prediction"
    )
    
    graph.add_node("process_case_summary", process_case_summary)
    graph.add_node("guideline_retrieval", guideline_retrieval_decision)
    graph.add_node("guideline_retrieve_content", retrieved_guideline_section_content)
    graph.add_node("guideline_based_predict_category", guideline_prediction_with_retry)
    
    graph.add_edge("process_case_summary", "guideline_retrieval")
    graph.add_edge("guideline_retrieval", "guideline_retrieve_content")
    graph.add_edge("guideline_retrieve_content", "guideline_based_predict_category")
    graph.add_edge("guideline_based_predict_category", END)
    
    graph.set_entry_point("process_case_summary")
    
    return graph.compile()

In [ ]:
# Initialize the guideline-only pipeline
print("Initializing guideline-only triage pipeline...")
guideline_only_triage_graph = build_guideline_only_triage_graph()
print("Guideline-only pipeline ready!")

In [ ]:
guideline_only_triage_graph

## Step 4: Evaluation Framework


The following sections implement a testing framework to evaluate the system's performance on real emergency cases. This enables:
1. Validating the quality of guideline retrieval
2. Assessing the relevance of retrieved similar cases
3. Analyzing the accuracy and reasoning of final triage assessments

### Single Case Testing

This function allows testing the system on individual cases to examine the detailed processing steps and outputs.

In [ ]:
def process_case_file_guideline_only(file_path: str) -> Dict[str, Any]:
    """
    Process a case file using the guideline-only triage assessment pipeline.
    
    Args:
        file_path (str): Path to the case file (json)
        
    Returns:
        Dict[str, Any]: Complete state from the guideline-only triage graph execution
    """
    # Load and format the case
    case_data = load_case_from_json(file_path)
    
    # Print the case content for inspection
    print("Case Content:")
    print(f"Age: {case_data.get('Demographics', {}).get('Age', 'Unknown')}")
    print(f"Sex: {case_data.get('Demographics', {}).get('Sex', 'Unknown')}")
    print(f"Chief Complaint: {case_data.get('Clinical Presentation', {}).get('Chief complaint', 'Unknown')[:100]}...")
    print("-" * 80)
    
    # Run the guideline-only triage system
    initial_state = GraphState(
        case_json_full=case_data,
        guideline_section_metadata=guideline_section_metadata
    )

    # Return the complete result state
    return guideline_only_triage_graph.invoke(initial_state)

In [ ]:
# process_case_file_guideline_only(file_path="PATH_TO_TEST_JSON")

### Batch Evaluation

This section implements a framework for processing multiple cases and saving the results for systematic evaluation. The outputs include:
1. Case summaries
2. Retrieved guideline sections
3. Complete triage assessments with reasoning

These structured outputs can be analyzed to evaluate the system's performance across different case types and triage categories.

In [ ]:
def batch_process_directory(
    input_directory: str,
    output_directory: str,
    max_files: int = None,
    verbose: bool = True
) -> int:
    """
    Process all JSON files using the guideline-only pipeline and save results.
    """
    # Create output directory
    os.makedirs(output_directory, exist_ok=True)
    
    # Find all JSON files
    json_files = []
    for root, _, files in os.walk(input_directory):
        for file in files:
            if file.endswith('.json'):
                json_files.append(os.path.join(root, file))
    
    if max_files:
        json_files = json_files[:max_files]
    
    print(f"Found {len(json_files)} JSON files to process with guideline-only pipeline")
    
    # Track statistics
    processed_count = 0
    skipped_count = 0
    error_count = 0
    
    # Process each file
    for file_path in tqdm(json_files, desc="Processing cases (guideline-only)"):
        try:
            # Get relative path for output structure
            rel_path = os.path.relpath(file_path, input_directory)
            file_name = os.path.basename(file_path)
            base_name = os.path.splitext(file_name)[0]
            
            # Create output subdirectory
            rel_dir = os.path.dirname(rel_path)
            output_subdir = os.path.join(output_directory, rel_dir)
            os.makedirs(output_subdir, exist_ok=True)
            
            # Check if output file already exists
            output_file_path = os.path.join(output_subdir, f"{base_name}_guideline_only_output.json")
            if os.path.exists(output_file_path):
                if verbose:
                    print(f"\nSkipping {rel_path} - guideline-only output file already exists")
                skipped_count += 1
                continue
            
            if verbose:
                print(f"\nProcessing with guideline-only pipeline: {rel_path}")
            
            # Process the case with guideline-only pipeline
            state = process_case_file_guideline_only(file_path)
            
            # Save output data
            output_data = {
                "summary_text": state.get("summary_text"),
                "case_json_full": state.get("case_json_full"),
                "selected_guideline_sections": state.get("selected_guideline_sections", []),
                "guideline_count": len(state.get("selected_guideline_sections", [])),
                "final_assessment": state.get("final_assessment"),
                "pipeline_info": {
                    "approach": "guideline_only",
                    "components": ["triage_definitions", "medical_guidelines"],
                    "excluded": ["past_cases", "specialty_prediction", "case_similarity"]
                }
            }
            
            # Save the output
            with open(output_file_path, 'w', encoding='utf-8') as f:
                json.dump(output_data, f, indent=2, ensure_ascii=False)
            
            processed_count += 1
            
            if verbose:
                print(f"Saved guideline-only output to: {output_file_path}")
                print(f"Guidelines retrieved: {len(state.get('selected_guideline_sections', []))}")
        
        except Exception as e:
            print(f"Error processing {file_path}: {str(e)}")
            print(traceback.format_exc())
            error_count += 1
    
    # Print summary
    print("\nGuideline-Only Processing Complete")
    print(f"Total files found: {len(json_files)}")
    print(f"Files processed: {processed_count}")
    print(f"Files skipped (already exist): {skipped_count}")
    print(f"Files with errors: {error_count}")
    
    return processed_count

In [ ]:
batch_process_directory(
    input_directory = "PATH_TO_INPUT_JSON_DIR",
    output_directory = "PATH_TO_OUTPUT_DIR",
    max_files = None,
    verbose = True
)